# 04 · Por Trás dos Panos (Caso A)

**Teoria**: docs/02-rdds-linhagem-particoes.md, docs/03-transformacoes-acoes-dag.md, docs/04-dataframes-catalyst-tungsten.md, docs/06-persistencia-e-otimizacao.md

🎯 **Objetivo deste notebook**: levantar o capô do Spark e explorar seus mecanismos internos — lineage de RDD, plano Catalyst, cache e comparação honesta com Pandas.

Você já confia nos resultados dos notebooks 01-03. Agora: **por que** o Spark levou aquele tempo pra calcular, e **como** ele decide o plano de execução?

In [ ]:
import sys

# Adiciona o diretório scripts/ ao path para importar módulos auxiliares do laboratório
sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

# Cria uma SparkSession local (Caso A — single-node, modo tutorial)
spark = get_local_session("04-por-tras-dos-panos")

## Lazy evaluation e lineage de RDD

💡 **Conceito fundamental**: nada abaixo executa até uma **ação** (`collect`, `sum`, ...) ser chamada.
Transformações (`filter`, `map`) só ficam registradas no grafo de lineage — o Spark as executa de forma **preguiçosa** (lazy evaluation) para poder otimizar o pipeline como um todo antes de disparar a execução.

⚠️ **Atenção**: se você chamar apenas `filter().map()`, nada acontece no cluster. O Spark só sai da inércia quando uma ação é invocada. Isso permite que o Catalyst enxergue o pipeline completo e decida a melhor ordem de execução.

In [ ]:
# Obtém o SparkContext de baixo nível (necessário para criar e manipular RDDs)
sc = spark.sparkContext

# Cria um RDD particionado em 8 fatias, cada uma com ~125.000 números
# numSlices controla o paralelismo: cada partition vira uma Task executada por um core
numbers = sc.parallelize(range(1, 1_000_001), numSlices=8)
print(f"Partições: {numbers.getNumPartitions()}")

# Transformações — são preguiçosas, nada roda ainda
evens = numbers.filter(lambda n: n % 2 == 0)    # Filtra apenas números pares
squared = evens.map(lambda n: n * n)             # Eleva cada par ao quadrado

# Ação — É AQUI que o Spark de fato constrói o DAG e executa
# O sum() força o Spark a percorrer todo o grafo de dependências
# Cada partition computa sua soma parcial, depois o driver reduz (soma) todos os parciais
total = squared.sum()
print(f"Soma dos quadrados dos pares de 1..1.000.000: {total:,}")

📌 **Observação sobre o resultado**:

O Spark executou o pipeline completo: `range → filter → map → sum`. Cada partition foi processada independentemente por uma Task, e o `sum()` agregou os resultados parciais (fase reduce).

💡 **Dica**: experimente variar `numSlices` (ex.: 2, 8, 64) e observe o efeito no tempo. Com poucos dados, muitas partições podem *piorar* a performance (overhead de agendamento).

⚠️ **Atenção**: `squared` ainda é um RDD — não podemos usar `.explain()` como faríamos com DataFrames.

In [ ]:
# O grafo de lineage que o Spark usaria para recomputar este RDD em caso de falha:
# toDebugString() mostra a genealogia como uma árvore indentada
# Cada linha com parênteses representa um RDD ancestral e a transformação aplicada
# Quanto mais profundo o grafo, mais caro seria recomputar após uma falha de nó
print(squared.toDebugString().decode())

🧠 **Entendendo o grafo de lineage**:

A árvore acima mostra a genealogia do RDD `squared`. Cada indentação representa uma dependência. Se uma partition for perdida (falha de nó worker), o Spark pode recomputá-la percorrendo essa linhagem — sem precisar de replicação.

📌 **Limitação**: o lineage de RDD é **opaco** — mostra apenas a sequência de transformações, sem otimização. É por isso que o DataFrame (próxima seção) é superior: ele expõe o schema ao Catalyst, que pode reordenar, podar e otimizar o plano de execução.

## RDD vs. DataFrame: o plano que o Catalyst enxerga

🎯 **Objetivo**: comparar o lineage opaco de RDD com o **plano Catalyst** de um DataFrame.

Mesma lógica de `filter → map → sum`, mas o DataFrame dá ao Spark um **schema** — é isso que torna a otimização do Catalyst possível.

📌 O `explain(True)` (modo estendido) mostra: **plano lógico não resolvido** → **plano lógico resolvido** → **plano físico** com os operadores reais que serão executados nos executores.

In [ ]:
# Cria um DataFrame com 1 milhão de números — há schema e tipagem, diferente do RDD
# .toDF("n") nomeia a única coluna como "n"
df = spark.range(1, 1_000_001).toDF("n")

# Filtra pares e calcula soma dos quadrados usando expressão SQL
# O Catalyst pode: empurrar o filtro (filter pushdown), podar colunas não usadas, etc.
resultado = df.filter(df.n % 2 == 0).selectExpr("sum(n * n) as soma_quadrados")

# explain(True) = extended mode: plano lógico + físico + otimizações aplicadas
# Compare a riqueza de informação com o toDebugString() do RDD acima
resultado.explain(True)
resultado.show()

📌 **Comparando os planos**:

Enquanto o lineage do RDD mostrava `MapPartitionsRDD[2] ← MapPartitionsRDD[1] ← ...`, o plano Catalyst do DataFrame revela operadores nomeados:

- `Scan` — leitura com estatísticas de tamanho
- `Filter` — predicate pushdown (quando aplicável)
- `HashAggregate` — agregação parcial + final (combine antes do shuffle)
- `Exchange` — shuffle dos dados entre partições (ponto mais caro do plano)
- `SerializeFromObject / DeserializeToObject` — serialização binária via Tungsten

💡 **Dica**: quanto mais informação o Catalyst tem (schema, estatísticas, dicas de broadcast), melhor o plano físico gerado.

## O Catalyst nos joins do notebook 03

📌 `BroadcastHashJoin` (sem Shuffle da tabela grande) vs. `SortMergeJoin` (Shuffle dos dois lados) — o mesmo par de junções do notebook 03, agora olhando o **plano de execução** por trás delas.

🧠 **Por quê?**: o tamanho das tabelas é o fator decisivo. `empresas` (~50 linhas) cabe na memória de cada executor e pode ser broadcastada. `funcionarios` (milhares de linhas) exige que ambos os lados sejam reparticionados por `id_funcionario` e shufflados entre os nós.

In [ ]:
from pyspark.sql.functions import broadcast

# Lê os dados das camadas Bronze — cada arquivo Parquet vira um DataFrame com schema inferido
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
funcionarios = spark.read.parquet(layer_path("local", "bronze", "funcionarios"))
empresas = spark.read.parquet(layer_path("local", "bronze", "empresas"))

# Broadcast join: dica explícita (broadcast()) para o Catalyst usar BroadcastHashJoin
# empresas (~50 linhas) é copiada para CADA executor — sem shuffle do lado grande vendas
join_broadcast = vendas.join(broadcast(empresas), "id_empresa")
print("--- broadcast join (empresas, 50 linhas) ---")
join_broadcast.explain()

# Shuffle join: SortMergeJoin — ambos os lados são shufflados por id_funcionario
# Funcionarios tem milhares de linhas, então não cabe no limite de broadcast (10 MB default)
join_shuffle = vendas.join(funcionarios, "id_funcionario")
print("--- shuffle join (funcionarios, milhares de linhas) ---")
join_shuffle.explain()

📌 **Interpretando os planos de join**:

No primeiro plano, procure por `BroadcastHashJoin`. A palavra "Broadcast" significa que a tabela `empresas` será copiada integralmente para TODOS os executores — o lado grande (`vendas`, centenas de milhares de linhas) não sofre shuffle.

No segundo plano, procure por `SortMergeJoin` e dois nós `Exchange`. Cada `Exchange` = um shuffle: um para `vendas` por `id_funcionario`, outro para `funcionarios` pelo mesmo campo. O custo de rede é significativamente maior.

⚠️ **Atenção**: Catalyst *pode* escolher BroadcastHashJoin automaticamente se a tabela pequena for menor que `spark.sql.autoBroadcastJoinThreshold` (10 MB por padrão).

## Spark SQL — mesmo motor, sintaxe SQL

💡 O Catalyst é o **mesmo otimizador**, independente da API usada (DataFrame, SQL ou RDD). Aqui usamos SQL puro — útil para equipes que já dominam SQL ou para migração gradual de data warehouses legados.

📌 Toda consulta SQL passa pelo mesmo pipeline: parser → análise lógica → Catalyst → plano físico → execução no cluster.

In [ ]:
# Registra DataFrames como tabelas temporárias — visíveis apenas nesta sessão Spark
# createOrReplaceTempView substitui a view se já existir com o mesmo nome
vendas.createOrReplaceTempView("vendas")
empresas.createOrReplaceTempView("empresas")
funcionarios.createOrReplaceTempView("funcionarios")

# Consulta SQL pura — o Catalyst resolve, otimiza e executa (mesmo plano que DataFrame API)
# Junta vendas → funcionarios → empresas, agrupa por setor, ordena por total e limita 10
resultado_sql = spark.sql("""
    SELECT e.setor, SUM(v.valor) AS total_vendas
    FROM vendas v
    JOIN funcionarios f ON v.id_funcionario = f.id_funcionario
    JOIN empresas e ON f.id_empresa = e.id_empresa
    GROUP BY e.setor
    ORDER BY total_vendas DESC
    LIMIT 10
""")
resultado_sql.show()

🧠 **O que aprender aqui**:

O resultado é idêntico ao que você obteria com DataFrame API. O Catalyst não diferencia a origem da consulta — ele sempre gera o mesmo plano físico otimizado.

💡 **Dica**: use `explain()` também em consultas SQL para depurar performance: `spark.sql("EXPLAIN SELECT ...").show(truncate=False)` mostra o plano completo sem executar.

## Cache: pagar uma vez, reusar muitas

🎯 **Objetivo**: demonstrar na prática como o cache evita reexecutar transformações caras.

O primeiro `count()` lê o Parquet do zero e aplica o filtro `valor > 50`. O segundo processamento (`groupBy ano.count`) roda sobre o dado já cacheado em memória — sem reler o disco.

⚠️ **Atenção**: cache só vale a pena se o mesmo DataFrame for reutilizado em múltiplas ações. Cachear um DataFrame usado uma única vez adiciona overhead desnecessário (serialização + memória).

In [ ]:
import time
from pyspark.sql.functions import col

# Aplica um filtro — transformação lazy, nada executou ainda
vendas_filtradas = vendas.filter(col("valor") > 50)

# Primeira ação: força leitura do Parquet + aplicação do filtro + contagem
# Ainda sem cache — toda ação futura reexecutará o pipeline do zero
start = time.perf_counter()
contagem_1 = vendas_filtradas.count()
sem_cache_segundos = time.perf_counter() - start

# Ativa cache (padrão: StorageLevel.MEMORY_ONLY) e materializa com um count()
# Após este count(), os dados filtrados residem em memória nos executores
vendas_filtradas.cache()
vendas_filtradas.count()  # materializa o cache

# Segunda consulta: groupBy ano — executa sobre o cache, sem reler o Parquet
start = time.perf_counter()
contagem_2 = vendas_filtradas.groupBy("ano").count().count()
com_cache_segundos = time.perf_counter() - start

print(f"Primeiro count() (sem cache ainda): {sem_cache_segundos:.3f}s")
print(f"groupBy sobre dado cacheado:         {com_cache_segundos:.3f}s")

# Libera a memória do cache — boa prática para não reter recursos desnecessariamente
vendas_filtradas.unpersist()

📌 **Análise dos tempos**:

Compare `sem_cache_segundos` vs `com_cache_segundos`.

O primeiro tempo inclui: leitura do Parquet no disco + descompressão + aplicação do filtro + contagem.
O segundo tempo inclui apenas: leitura da memória + groupBy + contagem.

💡 **Dica**: abra a aba **Storage** da Spark UI (http://localhost:4040/storage/) para ver o DataFrame cacheado, seu tamanho em memória e a fração de armazenamento utilizada.

## PySpark vs. Pandas nesta escala

🧠 **Por quê?** — em algumas centenas de milhares de linhas, o Pandas não paga overhead de distribuição/serialização — é bem provável que vença aqui.

A lição não é "Spark é lento", é **"use a ferramenta certa pro tamanho do dado"** (ver docs/05). O Spark brilha em datasets que não cabem na memória de uma única máquina.

📌 O Laboratório 13 revisit esta comparação em uma escala onde o Spark tem vantagem (centenas de milhões de linhas, múltiplos nós).

In [ ]:
# Converte o DataFrame Spark para Pandas — CUIDADO: traz todos os dados para o driver!
# Só é viável quando o dataset cabe na memória RAM do driver (não escala para grandes volumes)
pdf = vendas.toPandas()

# Pandas: groupBy + sum em memória local (single-node, dados em formato numpy nativo)
# Sem overhead de serialização, agendamento de Tasks ou shuffle de rede
start = time.perf_counter()
resultado_pandas = pdf.groupby("ano")["valor"].sum().sort_values(ascending=False)
pandas_segundos = time.perf_counter() - start

# Spark: mesmo cálculo, mas distribuído (overhead de scheduler + serialização + shuffle)
start = time.perf_counter()
resultado_spark = vendas.groupBy("ano").agg({"valor": "sum"}).collect()
spark_segundos = time.perf_counter() - start

print(f"Pandas groupBy: {pandas_segundos:.4f}s")
print(f"Spark groupBy:  {spark_segundos:.4f}s")
print()
print("Nesta escala, o menor overhead geralmente vence — o Lab 13 revisita")
print("essa mesma comparação numa escala onde a história do Spark muda.")

📌 **Conclusão da comparação**:

O Pandas venceu nesta escala (~500k linhas) porque:
1. **Sem serialização**: dados já estão em formato numpy na memória
2. **Sem agendamento**: não há Tasks para distribuir entre workers
3. **Sem shuffle**: tudo acontece em um único processo

💡 **Quando o Spark ganha?** Quando o dataset não cabe na memória de uma máquina (centenas de milhões a bilhões de linhas) ou quando o processamento exige múltiplos estágios de shuffle complexo.

⚠️ **Atenção**: `toPandas()` é perigoso em produção com datasets grandes — pode causar `OutOfMemoryError` no driver. Use apenas para amostras pequenas.

In [ ]:
# Encerra a SparkSession e libera todos os recursos (memória, threads, conexões JVM)
# Importante para não deixar processos órfãos no sistema
spark.stop()